In [4]:
11

11

In [5]:
import os
import random
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
import sentencepiece as spm

import evaluate
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets, disable_progress_bar
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    set_seed,
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score

2025-11-05 23:00:16.225094: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-05 23:00:17.212995: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-05 23:00:19.650788: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [6]:
def clear_gpu_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        print("🧹 GPU memory cleared")

In [7]:
print(f"📊 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

📊 Device: GPU
   GPU: NVIDIA GeForce RTX 4070 Ti SUPER
   Memory: 17.17 GB


In [8]:
# Replace with your HuggingFace dataset name
DATASET_PATH = "projected_datasets"
# Adjust these if your dataset has different column names
SOURCE_COLUMN = "shp"  # or "shipibo"
TARGET_COLUMN = "spa"   # or "spanish"

In [9]:
def load_full_dataset(dataset_path: str) -> DatasetDict:
    """
    Load Shipibo-Spanish parallel corpus with Sentiment labels from device
    
    Args:
        dataset_path: Path to the dataset file (e.g., "data/train.parquet")

    Returns:
        DatasetDict with train/validation/test splits
    """
    print(f"\n📥 Loading dataset: {dataset_path}")

    # Load the dataset
    dataset = load_dataset("parquet", data_files={"train": f"{dataset_path}/train.parquet",
                                             "test": f"{dataset_path}/test.parquet",
                                             "validation": f"{dataset_path}/validation.parquet"})

    # Display statistics
    print("\n📊 Dataset Statistics:")
    for split in dataset.keys():
        print(f"   {split}: {len(dataset[split])} examples")
        if len(dataset[split]) > 0:
            # Show first example
            example = dataset[split][0]
            print(f"   Example Shipibo: {example.get('shp', example.get('shipibo', 'N/A'))[:50]}...")
            print(f"   Example Spanish: {example.get('spa', example.get('spanish', 'N/A'))[:50]}...")
    
    return dataset


In [10]:
REGULAR_MODEL_PATH = "models/bert-sentiment-shipibo"
REGULAR_TOKENIZER_PATH = "tokenizers/xlm-roberta-base"

model = AutoModelForSequenceClassification.from_pretrained(REGULAR_MODEL_PATH, num_labels=3)
model.config.label2id = {'NEG': 0, 'NEU': 1, 'POS': 2}
model.config.id2label = {0: 'NEG', 1: 'NEU', 2: 'POS'}
tokenizer = AutoTokenizer.from_pretrained(REGULAR_TOKENIZER_PATH)

In [26]:
full_dataset = load_full_dataset(DATASET_PATH)


📥 Loading dataset: projected_datasets

📊 Dataset Statistics:
   train: 16505 examples
   Example Shipibo: Jato shinamawe mesko yokabo axon neskaakin....
   Example Spanish: Ahora hazles recordar a través de diferentes pregu...
   test: 2075 examples
   Example Shipibo: Kirikanin non raoki ika rabiti wishaxon axetixobon...
   Example Spanish: Escribe en un cuaderno un poema a nuestras plantas...
   validation: 1924 examples
   Example Shipibo: Metsara iwanke....
   Example Spanish: Fue maravilloso....


In [12]:
def compute_metrics(preds, labels):
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted"),
    }

In [28]:
def evaluate_model(model, tokenizer, dataset, batch_size=16, device=None):
    """Evaluate the model on the given dataset and print metrics."""
    model.eval()
    model.to(device or "cuda" if torch.cuda.is_available() else "cpu")
    device = next(model.parameters()).device
    
    labels_map = model.config.label2id
    dataset = dataset.map(lambda x: {"label": labels_map[x["sentiment_label"]]})
    
    # ⚙️ Tokenize dataset properly
    def preprocess(batch):
        return tokenizer(batch["shp"], truncation=True, padding=False)
    
    tokenized_ds = dataset.map(preprocess, batched=True)
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    dataloader = DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=DataCollatorWithPadding(tokenizer=tokenizer)
    )

    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    metrics = compute_metrics(preds, labels)
    return metrics

In [29]:
# Predecir usando dataset de validación
evaluate_model(model, tokenizer, full_dataset["validation"])

Map:   0%|          | 0/1924 [00:00<?, ? examples/s]

{'accuracy': 0.8856548856548857,
 'balanced_accuracy': 0.8437894411986933,
 'precision': 0.8865449855663363,
 'recall': 0.8856548856548857,
 'f1': 0.885787545764496}